# Learning process snapshots

This notebook plots generated samples from the same trained model at several checkpoints.  The goal is a simple visual for the poster: early checkpoints should look noisy or poorly structured, while later checkpoints should show sharper HI morphology.

Expected input files are produced by:

```bash
RUN_NAME=nf_fig2_u128_d2p15_noaug_200k \
EPOCHS=auto \
NUM_SAMPLES=6 \
MAX_PLOT_SAMPLES=5 \
SAMPLE_LABEL=dpm50 \
SAMPLER_CLASS=DPMSolverMultistepScheduler \
SAMPLER_STEPS=50 \
OVERWRITE=1 \
sbatch -A huterer2 scripts/slurm/sample_nf_generalize_epoch_snapshots.sbatch
```

The notebook does **not** resample. It only reads the saved `.npz` files and writes poster-ready PNGs.

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

RUN_NAME = 'nf_fig2_u128_d2p15_noaug_200k'
SAMPLE_LABEL = 'dpm50'
SNAPSHOT_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2' / 'epoch_snapshots'
OUT_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2' / 'quickcheck'
POSTER_DIR = PROJECT_DIR / 'poster' / 'figs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
POSTER_DIR.mkdir(parents=True, exist_ok=True)

print('project:', PROJECT_DIR)
print('snapshots:', SNAPSHOT_DIR)
print('poster:', POSTER_DIR)

In [ ]:
def load_sample_array(path: Path) -> np.ndarray:
    data = np.load(path)
    if isinstance(data, np.lib.npyio.NpzFile):
        for key in ('samples', 'images', 'arr_0'):
            if key in data:
                arr = data[key]
                break
        else:
            arr = data[data.files[0]]
    else:
        arr = data
    arr = np.asarray(arr)
    if arr.ndim == 4:
        if arr.shape[1] in (1, 3):
            arr = arr[:, 0]
        elif arr.shape[-1] in (1, 3):
            arr = arr[..., 0]
        else:
            raise ValueError(f'Cannot infer image channel axis for {path}: shape={arr.shape}')
    if arr.ndim != 3:
        raise ValueError(f'Expected samples with shape (N,H,W), got {arr.shape} from {path}')
    return arr.astype(np.float32)


def epoch_from_name(path: Path) -> int:
    m = re.search(r'_epoch(\d+)_', path.name)
    if not m:
        raise ValueError(f'Could not parse epoch from {path.name}')
    return int(m.group(1))


def discover_snapshot_files() -> list[Path]:
    pattern = f'{RUN_NAME}_epoch*_seed*_{SAMPLE_LABEL}.npz'
    files = sorted(SNAPSHOT_DIR.glob(pattern), key=epoch_from_name)
    if not files:
        raise FileNotFoundError(f'No snapshot files matched {SNAPSHOT_DIR / pattern}')
    return files

files = discover_snapshot_files()
rows = [(epoch_from_name(path), load_sample_array(path), path) for path in files]
print('found snapshots:')
for epoch, arr, path in rows:
    print(f'  epoch {epoch:04d}: {arr.shape}, {path.name}')

## Shared display scale

All panels use the same color scale based on the 1st--99th percentile across the selected snapshots. This makes the change with training time visible without one row rescaling itself independently.

In [ ]:
# Global image scale for fair visual comparison across epochs.
all_images = np.concatenate([arr for _, arr, _ in rows], axis=0)
VMIN, VMAX = np.percentile(all_images, [1.0, 99.0])
print('shared color scale:', VMIN, VMAX)

# Serif style matches the poster figures. Falls back cleanly if Times is unavailable.
POSTER_RC = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 22,
    'axes.titlesize': 24,
    'axes.labelsize': 24,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'figure.dpi': 140,
    'savefig.dpi': 300,
}

## Poster option A: learning grid

Rows are training checkpoints; columns are different noise seeds at the same checkpoint. This is useful when you want to show the full progression.

In [ ]:
def plot_epoch_grid(max_samples: int = 5, cmap: str = 'viridis') -> Path:
    n_rows = len(rows)
    n_cols = min(max_samples, max(arr.shape[0] for _, arr, _ in rows))
    with plt.rc_context(POSTER_RC):
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(2.35 * n_cols + 1.8, 2.35 * n_rows + 0.9),
            squeeze=False,
            constrained_layout=False,
        )
        for r, (epoch, arr, _) in enumerate(rows):
            for c in range(n_cols):
                ax = axes[r, c]
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
                ax.imshow(arr[c], cmap=cmap, vmin=VMIN, vmax=VMAX, interpolation='nearest')
                if r == 0:
                    ax.set_title(f'seed {c + 1}', pad=8, fontsize=22)
                if c == 0:
                    ax.set_ylabel(f'epoch {epoch}', rotation=0, ha='right', va='center', labelpad=54, fontsize=23)
        fig.suptitle('Generated HI maps during training', y=0.985, fontsize=30)
        fig.subplots_adjust(left=0.14, right=0.995, bottom=0.02, top=0.90, wspace=0.035, hspace=0.12)
        out = OUT_DIR / 'learning_process_epoch_grid_poster.png'
        poster_out = POSTER_DIR / 'learning_process_epoch_grid_poster.png'
        fig.savefig(out, bbox_inches='tight')
        fig.savefig(poster_out, bbox_inches='tight')
        plt.show()
    print('wrote', out)
    print('wrote', poster_out)
    return poster_out

grid_plot = plot_epoch_grid(max_samples=5)

## Poster option B: three-checkpoint panel

This is usually better for the poster because it is less crowded: early, middle, and final checkpoints, using the same noise seed for each row.

In [ ]:
def choose_three_rows(rows):
    if len(rows) <= 3:
        return rows
    idx = [0, len(rows) // 2, len(rows) - 1]
    return [rows[i] for i in idx]


def plot_epoch_triptych(sample_index: int = 0, cmap: str = 'viridis') -> Path:
    chosen = choose_three_rows(rows)
    with plt.rc_context(POSTER_RC):
        fig, axes = plt.subplots(1, len(chosen), figsize=(10.0, 3.55), constrained_layout=True)
        if len(chosen) == 1:
            axes = [axes]
        for ax, (epoch, arr, _) in zip(axes, chosen):
            ax.imshow(arr[sample_index], cmap=cmap, vmin=VMIN, vmax=VMAX, interpolation='nearest')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f'epoch {epoch}', pad=9, fontsize=25)
            for spine in ax.spines.values():
                spine.set_visible(False)
        fig.suptitle('Generated HI map sharpens during training', y=1.06, fontsize=30)
        out = OUT_DIR / 'learning_process_epoch_triptych_poster.png'
        poster_out = POSTER_DIR / 'learning_process_epoch_triptych_poster.png'
        fig.savefig(out, bbox_inches='tight')
        fig.savefig(poster_out, bbox_inches='tight')
        plt.show()
    print('wrote', out)
    print('wrote', poster_out)
    return poster_out

triptych_plot = plot_epoch_triptych(sample_index=0)

## Poster option C: one epoch at a time

If the poster needs three separate figures rather than one panel, this cell writes a small gallery for each checkpoint.

In [ ]:
def plot_individual_epoch_galleries(max_samples: int = 6, cmap: str = 'viridis') -> list[Path]:
    written = []
    with plt.rc_context(POSTER_RC):
        for epoch, arr, _ in rows:
            n = min(max_samples, arr.shape[0])
            n_cols = min(3, n)
            n_rows = int(np.ceil(n / n_cols))
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.8 * n_cols, 2.8 * n_rows + 0.45), squeeze=False)
            for i, ax in enumerate(axes.flat):
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
                if i < n:
                    ax.imshow(arr[i], cmap=cmap, vmin=VMIN, vmax=VMAX, interpolation='nearest')
                else:
                    ax.axis('off')
            fig.suptitle(f'epoch {epoch}', y=0.99, fontsize=28)
            fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.86, wspace=0.035, hspace=0.04)
            out = OUT_DIR / f'learning_process_epoch_{epoch:04d}_gallery.png'
            fig.savefig(out, bbox_inches='tight')
            plt.close(fig)
            written.append(out)
    for path in written:
        print('wrote', path)
    return written

individual_plots = plot_individual_epoch_galleries(max_samples=6)

## Quick display of saved poster files

In [ ]:
for path in [grid_plot, triptych_plot]:
    if Path(path).exists():
        display(Markdown(f'### `{Path(path).name}`'))
        display(Image(filename=str(path)))